In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split

print("imported the libraries!!!!")

mathematical represention of the sigmoid perceptron
input vector : x = [x1, x2, x3, .... xn]
weight vector : w = [w1, w2, w3, .... wn]
bias : b
summation function : z = w1*x1 + w2*x2 + w3*x3 + .... wn*xn + b
activation function : a = sigmoid(z) = 1 / (1 + exp(-z))


-> The range of the sigmoid function is between 0 and 1, which makes it suitable for binary classification problems.

the major drawback is Vanshing Gradient Problem :
- when the gradient becomes very small, in the backpropagation step, the weights of the earlier layers recieve very small updates,
which can lead to slow convergence or even prevent the model from learning effectively.


In [ ]:
import numpy as np

class SigmoidPerceptron:
    def __init__(self, input_size):
        # Initialize weights and bias randomly
        self.weights = np.random.randn(input_size)
        self.bias = np.random.randn()

    def sigmoid(self, z):
        # Sigmoid activation function
        return 1 / (1 + np.exp(-z))

    def predict(self, inputs):
        # Compute z = w·x + b
        weighted_sum = np.dot(inputs, self.weights) + self.bias

        # Return probability
        return self.sigmoid(weighted_sum)

    def fit(self, inputs, targets, learning_rate=0.01, epochs=100):
        num_samples, num_features = inputs.shape

        # Train for multiple passes through the dataset
        for epoch in range(epochs):
            for i in range(num_samples):

                # Select one training example
                input_vector = inputs[i]
                target = targets[i]

                # Forward pass
                prediction = self.predict(input_vector)

                # Compute prediction error
                error = target - prediction

                # Gradient computation
                gradient_weights = (
                    error
                    * prediction
                    * (1 - prediction)
                    * input_vector
                )

                gradient_bias = (
                    error
                    * prediction
                    * (1 - prediction)
                )

                # Update parameters
                self.weights += learning_rate * gradient_weights
                self.bias += learning_rate * gradient_bias

    def evaluate(self, inputs, targets):
        correct = 0

        for input_vector, target in zip(inputs, targets):

            # Predict probability
            prediction = self.predict(input_vector)

            # Convert probability to class label
            result = 1 if prediction >= 0.5 else 0

            # Count correct predictions
            if result == target:
                correct += 1

        # Compute accuracy
        accuracy = correct / len(targets)

        return accuracy

In [ ]:
import numpy as np
import math


class TanhPerceptron:
    def __init__(self, input_size):
        # Initialize weights and bias randomly
        self.weights = np.random.randn(input_size)
        self.bias = np.random.randn()

    def tanh(self, z):
        return math.tanh(z)

    def predict(self, inputs):
        # Compute weighted sum
        weighted_sum = np.dot(inputs, self.weights) + self.bias

        # Apply tanh activation
        return self.tanh(weighted_sum)

    def fit(self, inputs, targets, learning_rate=0.01, epochs=100):
        num_samples, num_features = inputs.shape

        for epoch in range(epochs):

            for i in range(num_samples):

                # Get one sample
                input_vector = inputs[i]
                target = targets[i]

                # Forward pass
                prediction = self.predict(input_vector)

                # Error
                error = target - prediction

                # Correct tanh derivative
                gradient = error * (1 - prediction**2)

                # Update weights
                self.weights += (
                    learning_rate
                    * gradient
                    * input_vector
                )

                # Update bias
                self.bias += (
                    learning_rate
                    * gradient
                )

    def evaluate(self, inputs, targets):
        correct = 0

        for input_vector, target in zip(inputs, targets):

            prediction = self.predict(input_vector)

            # Tanh threshold at 0
            result = 1 if prediction >= 0 else -1

            if result == target:
                correct += 1

        return correct / len(targets)

In [ ]:
import numpy as np


class ReLUPerceptron:
    def __init__(self, input_size):
        # Initialize weights and bias randomly
        self.weights = np.random.randn(input_size)
        self.bias = np.random.randn()

    def relu(self, z):
        return max(0, z)

    def relu_derivative(self, z):
        return 1 if z > 0 else 0

    def predict(self, inputs):
        # Compute weighted sum
        z = np.dot(inputs, self.weights) + self.bias

        # Apply ReLU activation
        return self.relu(z)

    def fit(self, inputs, targets, learning_rate=0.01, epochs=100):
        num_samples, num_features = inputs.shape

        for epoch in range(epochs):

            for i in range(num_samples):

                # Current sample
                input_vector = inputs[i]
                target = targets[i]

                # Forward pass
                z = np.dot(input_vector, self.weights) + self.bias
                prediction = self.relu(z)

                # Error
                error = target - prediction

                # Backpropagation
                gradient = error * self.relu_derivative(z)

                # Update weights
                self.weights += (
                    learning_rate
                    * gradient
                    * input_vector
                )

                # Update bias
                self.bias += (
                    learning_rate
                    * gradient
                )

    def evaluate(self, inputs, targets):
        correct = 0

        for input_vector, target in zip(inputs, targets):

            prediction = self.predict(input_vector)

            # Classification threshold
            result = 1 if prediction > 0 else 0

            if result == target:
                correct += 1

        return correct / len(targets)

In [ ]:
import numpy as np


class SoftmaxClassifier:

    def __init__(self, input_size, num_classes):

        self.weights = np.random.randn(
            input_size,
            num_classes
        ) * 0.01

        self.bias = np.zeros(num_classes)

    def softmax(self, z):

        z = z - np.max(z)

        exp_z = np.exp(z)

        return exp_z / np.sum(exp_z)

    def predict_proba(self, inputs):

        z = (
            np.dot(inputs, self.weights)
            + self.bias
        )

        return self.softmax(z)

    def predict(self, inputs):

        probabilities = self.predict_proba(inputs)

        return np.argmax(probabilities)

    def fit(
        self,
        inputs,
        targets,
        learning_rate=0.01,
        epochs=100
    ):

        num_samples = len(inputs)

        for epoch in range(epochs):

            for i in range(num_samples):

                x = inputs[i]

                y = targets[i]

                probabilities = self.predict_proba(x)

                target_vector = np.zeros(
                    len(self.bias)
                )

                target_vector[y] = 1

                error = (
                    target_vector
                    - probabilities
                )

                self.weights += (
                    learning_rate
                    * np.outer(x, error)
                )

                self.bias += (
                    learning_rate
                    * error
                )

    def evaluate(
        self,
        inputs,
        targets
    ):

        correct = 0

        for x, y in zip(inputs, targets):

            prediction = self.predict(x)

            if prediction == y:
                correct += 1

        return correct / len(targets)

In [ ]:
data = pd.read_csv("/Users/chandutammana/Desktop/DeepLearning/Datasets/diabetes.csv")
data.head()

In [ ]:
data['Outcome'].value_counts()

In [ ]:
class_0 = data[data['Outcome'] == 0]
class_1 = data[data['Outcome'] == 1]

# Undersample majority class
class_0 = class_0.sample(n=len(class_1), random_state=42)

# Create balanced dataset
data = pd.concat([class_0, class_1])

# Shuffle rows
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print(data['Outcome'].value_counts())
print(data.shape)

In [ ]:
# Separate input features and target variable
x = data.drop(columns='Outcome', axis=1)
y = data['Outcome']

# Split the dataset into training and testing sets
# test_size=0.2 -> 80% training, 20% testing
# random_state=123 -> ensures reproducible splits
# stratify=y -> preserves the class distribution in both sets
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

In [ ]:
# Create a StandardScaler object to standardize features
scaler = StandardScaler()

# Compute mean and standard deviation from the training data
# Then transform the training features to have mean=0 and std=1
x_train_scaled = scaler.fit_transform(x_train)

# Apply the same scaling parameters learned from the training data
# to the test data to prevent data leakage
x_test_scaled = scaler.transform(x_test)

In [ ]:
# Initialize the Sigmoid Perceptron with one weight per input feature
model1 = SigmoidPerceptron(input_size=x_train_scaled.shape[1])

# Train the model on the standardized training data
# learning_rate controls the step size of weight updates
# epochs specifies the number of complete passes through the training set
model1.fit(
    inputs=x_train_scaled,
    targets=y_train.values,
    learning_rate=0.01,
    epochs=100
)

# Evaluate the model's performance on the training data
train_accuracy = model1.evaluate(
    x_train_scaled,
    y_train.values
)

# Evaluate the model's ability to generalize to unseen data
test_accuracy = model1.evaluate(
    x_test_scaled,
    y_test.values
)

# Display the percentage of correctly classified training samples
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

# Display the percentage of correctly classified test samples
print(f"Testing Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
model2 = TanhPerceptron(input_size=x_train_scaled.shape[1])
model2.fit(
    inputs=x_train_scaled,
    targets=y_train.values,
    learning_rate=0.01,
    epochs=100
)

train_accuracy_tanh = model2.evaluate(
    x_train_scaled,
    y_train.values
)

test_accuracy_tanh = model2.evaluate(
    x_test_scaled,
    y_test.values
)

print(f"Tanh Perceptron Training Accuracy: {train_accuracy_tanh * 100:.2f}%")
print(f"Tanh Perceptron Testing Accuracy: {test_accuracy_tanh * 100:.2f}%")

In [ ]:
model3 = ReLUPerceptron(input_size=x_train_scaled.shape[1])
model3.fit(
    inputs=x_train_scaled,
    targets=y_train.values,
    learning_rate=0.01,
    epochs=100
)
train_accuracy_relu = model3.evaluate(
    x_train_scaled,
    y_train.values
)
test_accuracy_relu = model3.evaluate(
    x_test_scaled,
    y_test.values
)
print(f"ReLU Perceptron Training Accuracy: {train_accuracy_relu * 100:.2f}%")
print(f"ReLU Perceptron Testing Accuracy: {test_accuracy_relu * 100:.2f}%")

In [ ]:
model4 = SoftmaxClassifier(
    input_size=x_train_scaled.shape[1],
    num_classes=2
)
model4.fit(
    inputs=x_train_scaled,
    targets=y_train.values,
    learning_rate=0.01,
    epochs=100
)
train_accuracy_softmax = model4.evaluate(
    x_train_scaled,
    y_train.values
)
test_accuracy_softmax = model4.evaluate(
    x_test_scaled,
    y_test.values
)
print(f"Softmax Classifier Training Accuracy: {train_accuracy_softmax * 100:.2f}%")
print(f"Softmax Classifier Testing Accuracy: {test_accuracy_softmax * 100:.2f}%")

# Understanding Perceptrons and Sigmoid Perceptrons

## What is a Perceptron?

A perceptron is the simplest type of artificial neuron and serves as the fundamental building block of neural networks. It takes multiple input features, applies weights to them, adds a bias term, and produces an output.

Mathematically:

z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b

where:

- xᵢ = input features
- wᵢ = weights
- b = bias
- z = weighted sum

The goal of a perceptron is to learn appropriate values for the weights and bias so that it can make accurate predictions.

---

## Components Required by a Perceptron

### 1. Inputs

Inputs are the features that describe a data point.

Example:

```python
[5, 120, 30, 0.5]
```

These could represent attributes such as age, glucose level, BMI, etc.

### 2. Weights

Each input feature is assigned a weight that determines its importance.

```python
weights = [0.8, -0.3, 1.2, 0.1]
```

A larger weight means that feature has a stronger influence on the prediction.

### 3. Bias

The bias is an additional parameter that allows the decision boundary to shift independently of the input values.

```python
bias = 0.5
```

Without a bias term, the decision boundary would always pass through the origin.

### 4. Activation Function

The activation function converts the weighted sum into a prediction.

Common activation functions include:

- Step Function
- Sigmoid Function
- ReLU
- Tanh

---

# Traditional Perceptron

A traditional perceptron uses a step activation function.

Decision rule:

```python
if z >= 0:
    output = 1
else:
    output = 0
```

Characteristics:

- Produces only binary outputs (0 or 1).
- Cannot express uncertainty.
- Suitable only for linearly separable problems.
- Does not naturally produce probabilities.

Example outputs:

```text
0
1
0
1
```

This limitation led to the development of the sigmoid perceptron.

---

# What is a Sigmoid Perceptron?

A sigmoid perceptron replaces the step activation function with the sigmoid activation function.

Instead of producing hard classifications, it outputs probabilities.

Sigmoid Function:

σ(z) = 1 / (1 + e^(-z))

Output Range:

```text
0 < σ(z) < 1
```

Examples:

| z | Sigmoid Output |
|---|---|
| -5 | 0.0067 |
| -1 | 0.2689 |
| 0 | 0.5000 |
| 1 | 0.7311 |
| 5 | 0.9933 |

Interpretation:

```text
0.90 -> 90% confidence for class 1
0.10 -> 10% confidence for class 1
```

Because the sigmoid function is differentiable, gradient-based optimization becomes possible.

---

# Architecture of the Sigmoid Perceptron

```text
Input Features
       │
       ▼
Weighted Sum (w·x + b)
       │
       ▼
Sigmoid Activation
       │
       ▼
Probability Output
```

Example:

```text
Age = 25
BMI = 31
Glucose = 140

Weighted Sum = 2.5

Sigmoid(2.5) = 0.924
```

Prediction:

```text
92.4% probability of belonging to class 1
```

---

# Structure of the SigmoidPerceptron Class

The class consists of four primary methods:

## 1. __init__()

```python
def __init__(self, input_size):
```

Purpose:

- Initializes model parameters.
- Creates weights and bias.
- Prepares the neuron for learning.

Implementation:

```python
self.weights = np.random.randn(input_size)
self.bias = np.random.randn()
```

Initially, weights and bias are random because the model has not learned anything yet.

---

## 2. sigmoid()

```python
def sigmoid(self, z):
```

Purpose:

- Applies the sigmoid activation function.
- Converts any real number into a probability between 0 and 1.

Implementation:

```python
return 1 / (1 + np.exp(-z))
```

Examples:

```python
sigmoid(2)  = 0.8808
sigmoid(0)  = 0.5000
sigmoid(-2) = 0.1192
```

---

## 3. predict()

```python
def predict(self, inputs):
```

Purpose:

- Performs the forward pass.
- Computes the weighted sum.
- Applies the sigmoid activation.
- Returns the predicted probability.

Workflow:

```text
Inputs
   ↓
Weighted Sum
   ↓
Sigmoid
   ↓
Probability
```

Mathematically:

ŷ = σ(wᵀx + b)

Example:

```python
prediction = model.predict(sample)
```

Output:

```text
0.82
```

Meaning:

```text
82% probability of belonging to class 1
```

---

## 4. fit()

```python
def fit(self, inputs, targets, learning_rate, epochs):
```

Purpose:

- Trains the model.
- Updates weights and bias.
- Learns patterns from data.

Training Workflow:

```text
Forward Pass
    ↓
Prediction
    ↓
Error Calculation
    ↓
Gradient Calculation
    ↓
Parameter Update
    ↓
Repeat
```

### Step 1: Forward Pass

Compute prediction:

```python
prediction = self.predict(input_vector)
```

### Step 2: Error Calculation

```python
error = target - prediction
```

Example:

```text
Target = 1
Prediction = 0.75

Error = 0.25
```

### Step 3: Gradient Computation

The derivative of the sigmoid function is:

σ(z)(1 − σ(z))

In code:

```python
gradient_weights = (
    error
    * prediction
    * (1 - prediction)
    * input_vector
)

gradient_bias = (
    error
    * prediction
    * (1 - prediction)
)
```

### Step 4: Parameter Update

```python
self.weights += learning_rate * gradient_weights
self.bias += learning_rate * gradient_bias
```

The model gradually adjusts its parameters to reduce prediction errors.

---

## 5. evaluate()

```python
def evaluate(self, inputs, targets):
```

Purpose:

- Measures model performance.
- Computes classification accuracy.

Workflow:

### Generate Predictions

```python
prediction = self.predict(input_vector)
```

### Convert Probability to Class Label

```python
result = 1 if prediction >= 0.5 else 0
```

### Count Correct Predictions

```python
if result == target:
    correct += 1
```

### Compute Accuracy

```python
accuracy = correct / len(targets)
```

Formula:

Accuracy = Correct Predictions / Total Predictions

---

# What Optimization Method Does This Use?

The training loop processes one training example at a time:

```python
for i in range(num_samples):
```

After each sample, weights are immediately updated:

```python
self.weights += ...
self.bias += ...
```

Therefore, this implementation uses:

## Stochastic Gradient Descent (SGD)

Workflow:

```text
One Sample
     ↓
Compute Gradient
     ↓
Update Parameters
```

### Advantages

- Fast updates.
- Lower memory requirements.
- Works well on large datasets.

### Disadvantages

- Noisy updates.
- Less stable convergence than batch methods.

---

# Comparison with Other Gradient Descent Variants

## Batch Gradient Descent

```text
Entire Dataset
       ↓
Compute Average Gradient
       ↓
Single Update
```

Characteristics:

- Stable updates.
- Slower on large datasets.
- Requires more memory.

---

## Mini-Batch Gradient Descent

```text
Small Batch (32 or 64 samples)
          ↓
Compute Average Gradient
          ↓
Update Parameters
```

Characteristics:

- Most commonly used in deep learning.
- Faster than batch gradient descent.
- More stable than SGD.

---

## Stochastic Gradient Descent (Your Implementation)

```text
One Sample
     ↓
Compute Gradient
     ↓
Update Parameters
```

Characteristics:

- Fast updates.
- Efficient memory usage.
- More randomness during training.

---

# Why Standardization Is Important

Before training, features are standardized:

```python
scaler = StandardScaler()
```

Transformation:

x_scaled = (x − mean) / standard_deviation

Benefits:

- Features have similar scales.
- Faster convergence.
- More stable gradients.
- Better numerical performance.

Without scaling, features with large values may dominate the learning process.

---

# Traditional Perceptron vs Sigmoid Perceptron

| Feature | Traditional Perceptron | Sigmoid Perceptron |
|----------|----------|----------|
| Output | 0 or 1 | Probability |
| Activation Function | Step Function | Sigmoid |
| Differentiable | No | Yes |
| Supports Gradient Descent | No | Yes |
| Produces Confidence Scores | No | Yes |
| Suitable for Probabilistic Predictions | No | Yes |

---

# Relationship to Logistic Regression

The sigmoid perceptron is closely related to logistic regression.

Both use:

- Weighted sum
- Sigmoid activation
- Binary classification

Differences:

### Sigmoid Perceptron

- Often trained using simple gradient updates.
- Educational implementation.
- Demonstrates neural learning concepts.

### Logistic Regression

- Uses cross-entropy loss.
- More mathematically rigorous.
- Commonly used in production systems.

Conceptually:

```text
Traditional Perceptron
        ↓
Sigmoid Perceptron
        ↓
Logistic Regression
        ↓
Neural Networks
```

---

# Key Takeaways

- A perceptron is the simplest artificial neuron.
- It consists of inputs, weights, bias, and an activation function.
- Traditional perceptrons use a step function and output only 0 or 1.
- Sigmoid perceptrons use a sigmoid activation function and output probabilities.
- The sigmoid function is differentiable, enabling gradient-based learning.
- The `__init__()` method initializes model parameters.
- The `sigmoid()` method converts weighted sums into probabilities.
- The `predict()` method performs the forward pass.
- The `fit()` method trains the model by updating weights and bias.
- The `evaluate()` method measures classification accuracy.
- This implementation uses Stochastic Gradient Descent (SGD).
- The sigmoid perceptron serves as a bridge between traditional perceptrons and modern machine learning models such as logistic regression and neural networks.